# Quality-map uncertainty smoke

This pipeline validates the new two-mask experiment: optical content is degraded using a reference availability map, while the model receives an independently corrupted quality map. It also verifies pinned Sen1Floods11 SCL assets from both Earth Search and Planetary Computer.

**Smoke scores are pipeline checks, not paper evidence.**


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

WORKING = Path('/kaggle/working')
project = WORKING / 'geoai-ombria-robustness'
os.chdir(WORKING)
if project.exists():
    shutil.rmtree(project)
subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', 'codex/quality-map-uncertainty', 'https://github.com/NewRudy/geoai-ombria-robustness.git', str(project)],
    check=True,
)
os.chdir(project)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('source:', 'codex/quality-map-uncertainty', commit)


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'rasterio>=1.4'],
    check=True,
)
subprocess.run([sys.executable, 'scripts/ensure_cuda_compat.py'], check=True)
subprocess.run([sys.executable, 'scripts/check_cuda_runtime.py'], check=True)
subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    check=True,
)


In [ ]:
env = os.environ.copy()
subprocess.run(
    ['bash', 'scripts/run_quality_uncertainty_smoke.sh'],
    check=True,
    env=env,
)


In [ ]:
import hashlib, json
from zipfile import ZipFile
from IPython.display import FileLink, display

artifact = project / 'results' / 'ombria_quality_uncertainty_smoke_artifacts.zip'
with ZipFile(artifact) as archive:
    assert archive.testzip() is None
    manifest_name = next(
        name for name in archive.namelist()
        if name.endswith('/artifact_manifest.json')
    )
    manifest = json.loads(archive.read(manifest_name))
    root = manifest['root']
    for record in manifest['files']:
        name = root + '/' + record['path']
        assert hashlib.sha256(archive.read(name)).hexdigest() == record['sha256']
    smoke_name = next(
        name for name in archive.namelist()
        if name.endswith('/sen1floods11_scl_smoke.json')
    )
    scl_smoke = json.loads(archive.read(smoke_name))
    assert scl_smoke['status'] == 'pass'
    assert scl_smoke['manifest_match_fraction'] >= 0.95
print('artifact MB:', round(artifact.stat().st_size / 1024**2, 1))
print('files verified:', len(manifest['files']))
display(FileLink(str(artifact)))
